[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jeremydwong/bonelab_inverse_dynamics/blob/main/notebooks/muscle_force_validation.ipynb)

# Muscle-force validation, read critically: whether static or dynamic optimization gives muscle forces we can trust for femoral loading

I'm skeptical that static optimization (SO) or dynamic optimization gives muscle forces that are *right*, as opposed to forces that are plausible. So I went back to the three works people cite when they say "it's been validated": Prilutsky's cats (the only place individual muscle forces were measured and predicted in the same animal), the Grand Challenge competition to predict in vivo knee loads (blinded predictions against an instrumented knee), and Edwards, Miller & Derrick 2016 (static vs "dynamic" optimization, fed into a finite-element femur). For each one I ask two things. **A:** what does the validation actually show, and how good is the "truth" it's validated against? **B:** which is better, static or dynamic, and is that answer just an artefact of the parameters, cost function and muscle model they happened to pick?

> **How much to trust this page.** Every paper here was checked to exist by DOI, and the citation counts are from Semantic Scholar (October 2026). I read the full text of Fregly 2012, Kinney 2013 and Prilutsky & Zatsiorsky 2002 (PubMed Central author manuscripts) and of Edwards 2016 (the accepted manuscript on the University of Calgary repository). I could **not** get the full text of Prilutsky 1997 or Herzog & Leonard 1991 (ScienceDirect put up a robot check I can't, and shouldn't, click through), so for those I'm working from the abstracts plus the Prilutsky 1997 figure that Prilutsky & Zatsiorsky reproduce. Where a number comes from an abstract I say so. Numbers marked "my arithmetic" are mine, not the authors'.

This notebook reports on the following key findings about our reanalysis:

1. **Individual muscle forces have only been checked against measurement in the cat, and there SO gets the shape right but the size wrong.** In Prilutsky, Herzog & Allinger (1997) the best criterion (Dul's minimum fatigue) predicted soleus, gastrocnemius and plantaris forces with r = 0.97–0.99 but an average normalized error of 26–52 %. The criteria most people actually use (squared relative force, cubed stress, min/max) were off by 43–119 %, and linear ones by 59–322 % (all from the abstract). And the winning criterion has a free parameter per muscle (the slow-twitch fibre fraction), which they picked from a "physiologically reasonable" range for the trial they show.
2. **The cat "truth" is about as good as muscle-force truth gets, but it's a three-muscle problem.** Tendon force transducers measure the force in one tendon directly. The catch is that it's three synergists at one joint in a cat, and I couldn't confirm from the abstract whether the joint moments the optimizer had to reproduce came from ground reaction forces or from the measured muscle forces themselves. If it's the latter, the test is much easier than anything we do in humans. Herzog & Leonard (1991) and Herzog et al. (1993) found force sharing that shifts within a stride and across speeds, which no time-invariant cost can reproduce.
3. **The knee Grand Challenge validates contact force, not muscle force, and blinded errors were 0.3–0.5 body weights.** For the four winners of 2010–2012, average blinded RMS errors were 229–312 N medial and 238–326 N lateral, and total-force RMS errors were 372–440 N (Kinney 2013, Table 2). Lateral force variation was poorly predicted (blinded R² from 0.08 to 0.58 averaged over the two trials). The truth here is good (six-component load cells, R² > 0.997 on the bench), but the medial/lateral split is itself computed through a contact model, and the subjects are four TKA patients aged 68–86.
4. **Edwards 2016's "dynamic optimization" is a forward-dynamics tracking simulation, and its 32 % has no measured truth behind it.** The FD walking simulation (78 Hill muscles, bimodal block excitations, simulated annealing, cost = gait tracking + 0.2 × squared metabolic cost + a penalty for straying from textbook EMG on/off timing) is compared with SO (sum of squared muscle stresses, muscles as ideal force generators) run on the FD's *own* joint moments. Peak medial-surface minimum principal strain was −1271 µε (FD) vs −859 µε (SO), i.e. 32 %. Both were then compared with in vivo strain ranges from other people. Neither is validated.
5. **No study I found shows dynamic optimization gives more *valid* muscle forces than static.** Anderson & Pandy (2001) found them "practically equivalent" when the model and cost are matched. In the cat, SO agreed better with buckle forces than CMC (Karabulut 2020, abstract). At the hip, CMC over-predicted contact force the most, by up to 238 % vs in vivo values (Wesseling 2015). In the Grand Challenge, SO, dynamic optimization, EMG-driven and parametric methods all landed in the same error band. The static/dynamic split is mostly a proxy for *other* choices: the cost function, imposed EMG timing, the muscle model, and how much co-contraction each lets in.
6. **For the femur/AFF question, the muscles that matter most are the least constrained.** Edwards' FD vs SO difference sits in the gluteals (first-peak hip contact force 1824 N vs 1113 N, or 3.05 vs 1.86 BW by my arithmetic) and it changes lateral and medial shaft strain by roughly 250–412 µε. Real co-contraction is known to add up to ~1 BW at the knee (Trepczynski 2018). So I'd treat ±30 % on muscle-driven shaft strain as a *floor* on our uncertainty, and keep the net-joint-load results (which don't depend on a muscle solver) as the backbone of anything we claim.

In [1]:
# Colab bootstrap: a no-op when running locally. On Colab it clones the repo (for its
# data/ folder), installs boneid from the clone, and runs from notebooks/ so the
# relative paths below work.
import os, sys, subprocess
if "google.colab" in sys.modules and not os.path.exists("../src/boneid"):
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/jeremydwong/bonelab_inverse_dynamics"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "bonelab_inverse_dynamics"], check=True)
    os.chdir("bonelab_inverse_dynamics/notebooks")
os.makedirs("../reports", exist_ok=True)

In [2]:
# Display helper. The literature screenshots live in data/literature_figures/, which is
# gitignored: the repo is public and these are the publishers' figures. So the page links
# to them by relative path instead of embedding them. On a machine that has the folder
# (mine) the figure shows; anywhere else (GitHub, Colab) you get the citation and a note.
from pathlib import Path
from IPython.display import HTML, display

FIG_DIR = Path("../data/literature_figures")

def show_fig(name, caption, width=820):
    note = ("" if (FIG_DIR / name).exists() else
            '<p style="color:#a33;font-size:0.9em">[figure not included in the public repo '
            '(publisher copyright); see the paper via the DOI below]</p>')
    display(HTML(
        f'<figure style="margin:1em 0">{note}<img src="../data/literature_figures/{name}" '
        f'alt="Figure from the cited paper (not redistributed)" '
        f'style="max-width:100%;width:{width}px;border:1px solid #ccc">'
        f'<figcaption style="font-size:0.9em;color:#444;max-width:{width}px">{caption}</figcaption></figure>'))

## 1. Prilutsky's cats: the only direct test of individual muscle forces

### What was measured

Prilutsky, Herzog & Allinger (1997) put E-shaped tendon force transducers on soleus (SO), gastrocnemius (GA) and plantaris (PL) in three cats and filmed them walking and trotting on a treadmill at five nominal speeds from 0.4 to 1.8 m/s. They then predicted the three forces with static optimization on a three-segment (foot, shank, thigh), three-muscle hindlimb model, using six criteria from the literature. Everything below is from the abstract, plus the figure Prilutsky & Zatsiorsky (2002) reproduce from the paper.

| Criterion | Average normalized error (stance) | Correlation with measured |
|---|---|---|
| Linear: minimum force, minimum stress | 59–322 % | not given in abstract |
| Sum of squared relative forces; sum of cubed stresses; min upper-bound stress | 43–119 % | r > 0.9 for all three muscles |
| Minimum fatigue (Dul 1984, weighted by % slow-twitch fibres) | **26–52 %** | **0.97–0.99** |

Their conclusion is careful: SO "with the appropriate criterion can predict muscle forces adequately for specific movement conditions." That's a much narrower claim than the one people cite it for.

### How trustworthy is the truth?

Pretty good, as these things go, but with caveats I'd want to see addressed in the full text (which I couldn't get):

* **Transducer calibration.** A buckle or E-shaped transducer reads tendon strain-gauge output, which has to be converted to force with a calibration. How well that calibration holds in the living animal (tendon seating, cross-talk, drift over a session) sets the floor on the "truth". I didn't find a paper that puts a number on this for these transducers, so I can't give one.
* **Where did the joint moments come from?** SO needs the net joint moments as the constraint. The abstract mentions force transducers and high-speed video but no force plate. If the ankle and knee moments were computed as Σ (measured muscle force × moment arm), then the optimizer was handed the exact total and only had to *split* it. That's a fair test of a sharing rule, but it's a much easier problem than human SO, where the moments carry all the inverse-dynamics error. I can't confirm which it was without the methods section.
* **Three muscles, one-and-a-bit joints.** Two of the three muscles are uniarticular ankle extensors and GA crosses the knee too. That's the friendliest redundancy problem there is. Real human SO splits 40–80 muscles across a whole leg.
* **The error metric.** "Average normalized error" — I don't know what it's normalized by from the abstract. If it's normalized by the instantaneous force, low-force phases would inflate it. If it's by peak force, 26–52 % is a lot. Either way, the authors report it, so I'm taking it at face value.

In [3]:
show_fig("prilutsky1997_fig_cat_trot_SO_PL_GA_forces.png",
 "Measured (solid) and predicted (dotted, minimum fatigue criterion) forces of cat soleus (SO), plantaris (PL) and gastrocnemius (GA) "
 "over seven consecutive steps of trotting at 1.5 m/s. The slow-twitch fibre percentages used to make this prediction (SO 100 %, GA 20 %, PL 35 %) "
 "were chosen from a range of 'physiologically reasonable' values because the actual values for the trial were unknown. "
 "Originally Prilutsky BI, Herzog W, Allinger TL (1997) <i>J Biomech</i> 30:1025–1033, doi:10.1016/S0021-9290(97)00068-7, p. 1029; "
 "as reproduced in Fig. 4 of Prilutsky BI, Zatsiorsky VM (2002) Optimization-based models of muscle coordination, <i>Exerc Sport Sci Rev</i> 30(1):32–38, "
 "doi:10.1097/00003677-200201000-00007 (PMC2821033 author manuscript). (reproduced for scholarly commentary; © the publisher)", width=640)

### Results, and my read

The figure is the best case, and it's honestly impressive: seven steps, three muscles, and the dotted lines sit almost on top of the solid ones. Peaks are around 25 N (SO), 35–40 N (PL) and 55–60 N (GA). But three things temper it for me.

**First, the free parameter.** The winning criterion weights each muscle by its slow-twitch fibre fraction. Prilutsky & Zatsiorsky say outright that the fraction "engaged in the force development during this trial was unknown", so they tried different physiologically reasonable values (their reference to Table 2 of the 1997 paper) and show the figure with SO 100 %, GA 20 %, PL 35 %. So the best fit has three knobs that were set with the measurement in hand. That isn't cheating (they were testing whether *any* reasonable setting works), but it isn't a blind prediction either.

**Second, the criteria everyone actually uses don't do this well.** OpenSim's default SO minimizes the sum of squared activations, which is close to the "squared relative force" / "cubed stress" family. In the cat that family gave 43–119 % average error. That's with a three-muscle model and cat-specific anatomy.

**Third, the same lab found the opposite with a different framing.** Herzog & Leonard (1991) compared linear and nonlinear optimization predictions against SO, GA and PL forces in one cat across locomotor tasks, and the abstract says plainly the predictions "did not agree well". Their reason: force sharing changes *within* a step cycle and *across speeds*, and the models ignored both. Herzog, Leonard & Guimaraes (1993; 10 cats) showed why that's hard: GA and PL forces rise with speed or resistance but peak soleus force stays nearly constant, with big stride-to-stride modulation. They read that as central control, not a moment-driven rule. Prilutsky, Herzog & Allinger (1994) showed the SO-vs-GA force-sharing plot is a *loop* within each step (counter-clockwise), and the loop narrows with speed. Any rule of the form "forces = f(joint moments at this instant)" has to get that loop from the knee moment through GA alone.

So I'd summarize the cat as: **right shape, wrong size, and the size depends on a parameter you don't know.** The newer cat test (Karabulut et al. 2020: OpenSim, 13 DoF, 25 Hill muscles, buckle forces on soleus, MG and tibialis anterior) says predictions were "consistent" with measurement and SO agreed more closely than computed muscle control (CMC). I couldn't get the error numbers from the abstract.

(Prilutsky, Herzog & Leonard 1996 is sometimes cited alongside these; as far as I can tell from its title and record it's about energy transfer between the ankle and knee by GA and PL, not a test of optimization, so I've left it out of the argument.)

## 2. The knee Grand Challenge: blinded predictions against an instrumented knee

### What was measured

Fregly et al. (2012) set up an annual competition. Each year they released the full gait-lab data of one patient with a force-measuring tibial prosthesis (markers, force plates, 14–15 channels of EMG, strength, CT/MR/x-ray, implant geometry, OpenSim models) *except* the measured contact forces for two chosen gait trials. Competitors predicted medial and lateral tibial contact force blind. The first two subjects were both 83-year-old men (JW: 68 kg, neutral alignment; DM: 70 kg, valgus); later ones were PS (86, male, 75 kg) and SC (68, female, 79 kg).

### How trustworthy is the truth?

This is the best "truth" in the whole area, and I think it's honest to say so. But it's a truth about *contact force in an artificial knee*, not muscle force:

* **Accuracy of the sensor.** The second-generation tibial tray measures all six load components with bench accuracy R² > 0.997 (Kirking 2006). The first generation (JW) used four uniaxial load cells at the corners of the tray (D'Lima 2005).
* **The medial/lateral split is derived.** Fregly 2012 says both devices "permit calculation of medial and lateral contact force through the use of deformable contact models." So the medial/lateral "measurement" competitors were scored against already has a model in it. The total is the cleanest quantity.
* **They're TKA patients.** No menisci, a polyethylene insert, often no ACL, implant geometry instead of cartilage, and 68–86-year-olds walking how they walk after surgery. Fregly 2012 says themselves that prediction in natural knees "will be more difficult."
* **Muscle forces are validated only indirectly.** Fregly 2012 is explicit: implant data give "direct validation of estimated articular contact forces and indirect validation of estimated muscle forces." Many muscle-force patterns can give the same contact force.
* **The trials were picked to be hard.** Kinney 2013 says the competition trials were chosen so the measured forces were "in some way unique or unexpected" (e.g. trunk sway, medial thrust). That's good for testing, but it means the numbers below are worse than you'd get on plain walking.

In [4]:
show_fig("fregly2012_fig2_jw_medial_lateral_contact_four_gaits.png",
 "Medial and lateral tibial contact force (body weights) measured by the instrumented prosthesis of subject JW for normal, medial thrust, "
 "walking pole and trunk sway gait; grey bands are the range over 5 trials. Fig. 2 of Fregly BJ, Besier TF, Lloyd DG, Delp SL, Banks SA, Pandy MG, "
 "D'Lima DD (2012) Grand challenge competition to predict in vivo knee loads, <i>J Orthop Res</i> 30(4):503–513, doi:10.1002/jor.22023 "
 "(PMC4067494 author manuscript). (reproduced for scholarly commentary; © the publisher)")

The thing I take from Fregly's figure: in normal gait JW's medial compartment peaks around 1.8 BW and lateral around 1 BW, and gait changes like medial thrust or trunk sway move them by only a few tenths of a BW. That's the size of effect a model has to resolve. Fregly's Table 1 also makes a point I didn't expect them to make so bluntly: published models predict peak total knee force mostly in the 3.0–3.5 BW range, "about 1 BW larger than typical in vivo measurements" of 2.0–2.5 BW. And in their discussion: "it is unlikely that current methods provide the correct solutions."

### Results (Kinney et al. 2013)

Kinney reports the four winners of 2010–2012. The methods were all over the map: Kim et al. (2010) and Hast & Piazza (2011) used deformable contact models; Manal & Buchanan (2012a) used an EMG-driven model; Knowlton et al. (2012b) used a parametric model that predicts a *space* of feasible solutions. Over all 11 teams, competitors used static and dynamic optimization, EMG-driven and parametric methods.

Average over the two competition trials, **blinded** (Kinney 2013, Table 2):

| Winner | Medial RMSE (N) / R² | Lateral RMSE (N) / R² | Total RMSE (N) / R² |
|---|---|---|---|
| 2010 (Kim et al.) | 229 / 0.82 | 326 / **0.08** | 440 / 0.73 |
| 2011 (Hast & Piazza) | 312 / 0.50 | 301 / 0.30 | 432 / 0.57 |
| 2012a (Manal & Buchanan, EMG-driven) | 294 / 0.82 | 238 / 0.55 | 438 / 0.81 |
| 2012b (Knowlton et al., parametric) | 308 / 0.83 | 242 / 0.58 | 372 / 0.82 |

In 2012 they also asked for **unblinded** predictions after the measurements were released. 2012b improved to 229 / 155 / 280 N (medial / lateral / total). 2012a improved medial and total (210 and 395 N) but its lateral R² *fell* from 0.55 to 0.29. That blinded→unblinded gap is roughly the "tuning budget": how much of a good fit in a non-blinded paper comes from knowing the answer.

By my arithmetic, with the subjects weighing 68–79 kg (667–775 N), the blinded medial and lateral RMS errors are about **0.3–0.5 BW**, on compartment forces that peak around 1–2 BW.

In [5]:
show_fig("kinney2013_fig2_blinded_predictions_vs_measured.png",
 "Medial, lateral and total contact forces for the two trials of each competition: measured (black), blinded prediction (dark grey dashed) and, "
 "in 2012, unblinded prediction (light grey). Note the 0–4000 N y-axis, which visually shrinks errors of 200–400 N. Fig. 2 of Kinney AL, Besier TF, "
 "D'Lima DD, Fregly BJ (2013) Update on grand challenge competition to predict in vivo knee loads, <i>J Biomech Eng</i> 135(2):021012, "
 "doi:10.1115/1.4023255 (PMC3597120). (reproduced for scholarly commentary; © the publisher)")

### Our read

The Grand Challenge is the cleanest evidence there is, and I think the honest summary is:

* **Total force: OK-ish. Medial: decent shape. Lateral: mostly not predicted.** A lateral R² of 0.08 from a competition *winner* is sobering.
* **The method family didn't decide the winner.** Kinney: "completely different modeling methods" gave comparable accuracy, and two 2012 co-winners (EMG-driven vs parametric) neither of which used a contact model beat the earlier contact-model winners on some measures. So "EMG-driven is better" isn't what the competition showed. That claim comes from later, non-blinded studies (Bennett 2022: RMSE 192–674 N for SO vs 152–487 N for calibrated EMG-informed, in 3 TKA subjects).
* **What does move the error is geometry and calibration, not static vs dynamic.** Later work on these and similar data found: subject-specific alignment and contact points took peak-1 errors from 63.1 % / 42.0 % (medial / lateral, generic) down to 12.4 % / 11.9 % with SO (Lerner 2015); uncalibrated vs calibrated EMG-driven models gave 323 / 348 N vs 52.6 / 56.6 N RMSE, but the calibration used the measured contact force (Serrancolí 2016); synergy-constrained optimization gave 83–161 N vs 124–343 N for independent controls (Walter 2014); a generic OpenSim SO model on the CAMS-Knee patients under-predicted walking force with RMS 47.6 % BW and *over*-predicted squat by 105.7 % BW (Imani Nejad 2020). These are mostly from abstracts (via my earlier notes), and most are not blind.
* **Co-contraction is real and SO suppresses it.** In nine instrumented-knee patients, quadriceps–gastrocnemius co-contraction added up to ~1 BW (about 50 %) to the late-stance peak in walking (Trepczynski 2018). Minimum-effort SO is built to avoid exactly that.

## 3. Edwards, Miller & Derrick (2016): static vs "dynamic" optimization into a finite-element femur

### What was actually done

I read the accepted manuscript and its supplement, so I can pin this down.

**The musculoskeletal model.** 3D, 10 segments, 9 joints, 78 Hill-type muscles (20 per leg crossing the hip or attached to the femur), parameterized as a young adult woman (1.65 m, 61.0 kg). Muscle parameters started from Arnold et al. (2010) (a cadaver-based model) and were adjusted so joint strength matched average young-female dynamometry. Viscoelastic/Coulomb foot–ground contact.

**The "dynamic optimization" is a forward-dynamics (FD) tracking simulation.** One stride of periodic, bilaterally symmetric walking at 1.25 m/s. Each muscle's excitation was a **bimodal block signal with six parameters** (two magnitudes, two on-times, two off-times). A parallel simulated-annealing algorithm (SPAN, Higginson 2005) adjusted them to minimize (from the supplement):

$$J = J_\text{track} + 0.2\,J_\text{metcost} + 50\,J_\text{exc}$$

* $J_\text{track}$: mean squared deviation from 15 target time series (3D pelvis, 3D lumbar, 3D hip angles, knee and ankle angle, 3D GRF and the knee adduction moment), each normalized by the between-subject SD of 14 people walking "normally" (Miller 2014 data). Not the same person as the model.
* $J_\text{metcost}$: the *square* of gross metabolic cost of transport (Umberger 2003 muscle energy model, plus 1 W/kg resting).
* $J_\text{exc}$: penalty on each muscle's on/off timing straying from textbook indwelling-EMG timing (Sutherland 2001). Excitation timing for larger muscles was also *constrained* to be similar to that EMG.

The result walked at 1.25 m/s at 3.64 J/m/kg, with kinematics and GRF "always within two standard deviations" of the experimental means.

**The SO.** Inverse dynamics on the *FD simulation's own* motion and GRF (so experimental error is removed, like Anderson & Pandy 2001), then at each instant: reproduce all joint moments exactly, minimize the **sum of squared muscle stresses** (Glitsch & Baumann 1997), interior-point solver. Muscles were **ideal force generators**, with no force–length–velocity, no activation dynamics, no tendon, no passive force.

**The FE femur.** From the VAKHUM database: CT of a **99-year-old female cadaver** (155 cm, 55 kg), scaled to the model (length, then radius from bone mass ∝ body mass), density bumped up 26 % to match a young woman, E = 6850 ρ^1.49, ~105 k hex elements. Muscle and hip contact forces were applied as point loads at 10 % increments of stance, plus the two hip-contact-force peaks (JCF1, JCF2).

### Where the 32 % comes from

Peak minimum principal strain (compression, medial shaft surface): FD −1271 µε vs SO −859 µε. (1271 − 859) / 1271 = 32 %. The maximum principal strain (tension) was 945 vs 752 µε, which is a 20 % difference by my arithmetic. So "up to 32 %" is the compressive side; the tensile side, which is the one I care about for AFF, differs by about 20 %.

Other numbers from the results:

* Hip contact force at JCF1 (~30 % stance): FD 1824 N vs SO 1113 N. At JCF2: 1410 vs 1406 N, basically identical.
* At the proximal lateral femur (35 mm below the greater trochanter, where Aamodt measured), at 30 % stance, longitudinal and transverse strains were ~37 % larger with FD (278 and 85 µε); principal strain angle never differed by more than 5°.
* Along the shaft at JCF1: peak lateral εmax differed by ~250 µε and peak medial εmin by 412 µε; anterior/posterior differences ≤ 113 µε. At JCF2 the lateral/medial differences dropped to ~100 / 119 µε, but posterior εmin still differed by 171 µε even though the hip contact forces were the same.

In [6]:
show_fig("edwards2016_fig3_hip_contact_and_muscle_forces_FD_vs_SO.png",
 "Resultant hip contact force and forces of selected femur-attached muscles over stance, forward-dynamics simulation (FD, black) vs static "
 "optimization (SO, grey). GMAX/GMED/GMIN = gluteus maximus/medius/minimus, ADDMAG = adductor magnus, RECFEM = rectus femoris, VAS = vasti, "
 "HAM = hamstrings, GAS = gastrocnemius. Fig. 3 of Edwards WB, Miller RH, Derrick TR (2016) Femoral strain during walking predicted with muscle "
 "forces from static and dynamic optimization, <i>J Biomech</i> 49(7):1206–1213, doi:10.1016/j.jbiomech.2016.03.007; rendered from the accepted "
 "manuscript (University of Calgary PRISM, doi:10.11575/PRISM/34235). (reproduced for scholarly commentary; © the publisher)")

Looking at their Fig. 3 myself: the first-half-of-stance gap is mostly **gluteus medius** (FD ~480 N vs SO ~200 N around 30 % stance), **gluteus minimus** (FD ~200 N vs SO ~20 N around 15 %) and **gluteus maximus** (FD ~140 N vs SO ~50 N in early stance). FD also carries much more rectus femoris, hamstrings and gastrocnemius through stance, i.e. more co-contraction around the knee and ankle. One thing I can't square: the text says SO had "virtually no co-contraction of the hip adductors compared to FD" in the first half of stance, but the one adductor they plot (adductor magnus) shows SO with ~160 N at heel strike falling to zero by ~20 %, and FD with *zero* until ~63 % stance (then a ~170 N spike). Maybe other adductors not shown carry the FD co-contraction; I can't tell from the paper. I'd trust the gluteal story more than the adductor one.

In [7]:
show_fig("edwards2016_fig7_femoral_shaft_strain_paths_FD_vs_SO.png",
 "Maximum principal strain along the lateral and anterior surfaces, and minimum principal strain along the medial and posterior surfaces, "
 "along the femoral shaft at the first (top) and second (bottom) peak of hip contact force; FD (black) vs SO (grey). Fig. 7 of Edwards WB, "
 "Miller RH, Derrick TR (2016) <i>J Biomech</i> 49(7):1206–1213, doi:10.1016/j.jbiomech.2016.03.007; rendered from the accepted manuscript. "
 "(reproduced for scholarly commentary; © the publisher)", width=640)

### How trustworthy is the truth? There isn't one.

Edwards et al. are upfront about this, which I appreciate. "A general limitation of this study is that we cannot affirm which method of muscle force estimation is more accurate." The only measurement in the paper is a *literature* comparison with Aamodt et al. (1997), a strain-gauge rosette on the proximal lateral femur of a 49-year-old woman having surgery for snapping hip. Aamodt measured axial strain up to 1,133 µε; FD gave 745 µε and SO 467 µε at that site. The authors call a rigorous comparison "a futile exercise" (different person, age, geometry, speed, unknown gauge location), and they note that moving the virtual gauge a few millimetres changes the strain by 100–200 µε. So the "agreed well with in vivo" in the abstract means: same tension/compression ratio and principal-strain angle, roughly the same order of magnitude.

Also worth saying out loud: the bone is a 99-year-old cadaver femur, scaled and re-densified to look young, under a generic (not subject-specific) model, with tracking targets from 14 *other* people. Every one of those choices could move absolute strain by tens of percent. The authors themselves cite near two-fold differences between common density–modulus relationships (Schileo et al. 2007, as cited by Edwards).

### Our read

This is a good **sensitivity study**, not a validation. Its real message is: *two defensible muscle solvers, given identical joint kinematics and joint moments, disagree by 20–32 % on peak femoral shaft strain.* That's a lower bound on the muscle-solver part of the uncertainty, because they share the same model, muscle geometry and bone.

## 4. Static vs dynamic: which is better, and is the answer an artefact?

Short version: **the evidence can't decide, and most of the reported differences are not really static-vs-dynamic.** Here is what each comparison actually changed, besides the label.

**Edwards 2016 (FD > SO by 20–32 % in strain).** The FD and SO differ in at least four ways at once:

1. *Cost function.* FD minimized tracking error + 0.2 × (metabolic cost)² + an EMG-timing penalty. SO minimized squared muscle *stress*. These favour different muscles regardless of time.
2. *Imposed timing.* FD's large-muscle excitations were *constrained* to textbook EMG on/off times, and each excitation is a block with only two magnitudes. If the textbook says gluteus medius is on through early stance, it's on, even if the moment doesn't need it. That's co-contraction injected by parameterization, not discovered by dynamics.
3. *Muscle model.* FD had Hill muscles with force–length–velocity, activation dynamics and (presumably) passive force; SO used ideal force generators with no limits besides stress. Activation dynamics alone forces muscles to stay on longer and switch off slower, which shows up as co-contraction.
4. *Search.* Simulated annealing over 6 × 78 = 468 parameters gives *a* good solution, not necessarily the global optimum; a slightly different run could land on different co-contraction.

So the 32 % tells me how much shaft strain depends on these modelling choices. It doesn't tell me dynamic is right.

**Anderson & Pandy (2001): practically equivalent.** When the same model, the same cost (they used metabolic energy in DO and a matching cost in SO) and the same motion are used, SO and DO muscle forces and joint contact forces were "remarkably similar". This is the clean static-vs-dynamic test, and the answer was "no difference". It also has no ground truth.

**Cat, SO vs CMC (Karabulut 2020).** SO agreed more closely with buckle forces than CMC (abstract; no numbers available to me). Here the "dynamic" method (CMC, which is SO with activation and contraction dynamics, tracking kinematics) was *worse*.

**Hip, SO vs CMC (Wesseling 2015).** CMC gave the highest hip contact forces (median 3.9 × peak GRF) and was up to 238 % larger than in vivo literature values; SO (3.0–3.1 ×) "approached experimental values best". Same direction as Edwards (dynamic → more co-contraction → more joint load), but here the extra load looks *too high*.

A rough cross-check of Edwards against the hip implants (my arithmetic, so take it as a sanity check, not a result): with the model's 61.0 kg (598 N), FD's first-peak hip force of 1824 N is 3.05 BW, SO's 1113 N is 1.86 BW, and both second peaks (~1410 N) are 2.35 BW. Bergmann et al. (2001) measured an average walking peak of 238 % BW in four THA patients. So SO's second peak lands right on it, FD's first peak is about 0.7 BW high and SO's first peak about 0.5 BW low. (Different people, older patients, likely slower walking, so I wouldn't push this far.)

**Knee, Grand Challenge.** Static and dynamic optimization, EMG-driven and parametric methods all gave blinded errors in the same band (229–326 N per compartment). The things that moved errors by factors of 3–6 were geometry (Lerner 2015) and calibration with the answer in hand (Serrancolí 2016).

**Cat criteria (Prilutsky 1997).** Within *static* optimization alone, changing the criterion moved the average error from 26 % to 322 %. That's a far bigger effect than any static-vs-dynamic difference reported anywhere above.

### Where the evidence simply can't decide

* No human study compares SO or DO predicted muscle forces against measured tendon forces. Achilles forces have been measured (Komi 1990; Finni 1998), and in one cyclist the measured triceps surae moment was about 65 % of the net ankle moment (Gregor 1991), but nobody ran SO against it that I could find.
* Contact force validation (hip, knee) is indirect: many muscle patterns give the same contact force, especially at the hip where abductors dominate.
* Bone strain validation of either method doesn't exist for the femoral shaft.
* Every head-to-head comparison changes the cost function and muscle model along with the static/dynamic label, so the label's own effect is never isolated (except in Anderson & Pandy, where it was ~zero).

So my answer to B: **there's no evidence that dynamic optimization gives more valid muscle forces than static optimization.** There is good evidence that the *cost function, muscle parameters, imposed timing and joint geometry* each change the answer by tens of percent, and that these choices, not static vs dynamic, are what separate published results.

## 5. What this means for our femur/AFF question

* **Shaft bending is set by exactly the muscles we know least about.** Gluteals, adductors and the ITB set femoral shaft bending (Duda 1998 showed adding adductors and ITB to hip contact + abductors cut the FE strain difference from 26 % to 5 %). Edwards' FD/SO disagreement is concentrated in the gluteals. Hip contact force is not very sensitive to how the abductors share load, which is why hip validation can look good while shaft strain is still uncertain.
* **Treat ~20–30 % as a floor on muscle-driven strain uncertainty.** That's the spread between two careful solvers sharing *everything* else. Add generic geometry, unknown co-contraction (up to ~1 BW at the knee, Trepczynski 2018) and bone material models, and I'd expect more.
* **Lead with the net joint loads.** Our pipeline's inverse dynamics doesn't need a muscle solver, and the bowing counterfactuals in the AFF notebook are about geometry changing the moment arm of loads we *measure*. Those comparisons (same gait, different skeleton) are much less exposed to the solver problem than absolute strains.
* **If we do add muscles, bracket rather than pick.** Run a family of solutions (different SO exponents, with and without activation dynamics, an upper-co-contraction case) and report the band. If a conclusion holds across the band, it's probably real. If it flips, we've learned the data can't decide.
* **The validation targets that exist for the femur:** hip contact force (OrthoLoad/HIP98), shaft axial force and bending from distal femoral replacements (Taylor & Walker 2001: varus–valgus bending peaked at 8.5–9.8 BW·cm in walking), and proximal-lateral strain in two patients (Aamodt 1997). None of these is a healthy shaft in a young person, and none tests muscle forces directly.

## References

Citation counts from Semantic Scholar, retrieved 2026-10-23. All DOIs resolved. "Full text" means I read it; otherwise I worked from the abstract.

**Cat tendon-force validation**

* Prilutsky BI, Herzog W, Allinger TL (1997). Forces of individual cat ankle extensor muscles during locomotion predicted using static optimization. *J Biomech* 30:1025–1033. doi:10.1016/S0021-9290(97)00068-7 — 57 (abstract + reproduced figure)
* Prilutsky BI, Zatsiorsky VM (2002). Optimization-based models of muscle coordination. *Exerc Sport Sci Rev* 30(1):32–38. doi:10.1097/00003677-200201000-00007 — 267 (full text, PMC2821033)
* Prilutsky BI, Herzog W, Allinger TL (1994). Force-sharing between cat soleus and gastrocnemius muscles during walking: explanations based on electrical activity, properties, and kinematics. *J Biomech* 27:1223–1235. doi:10.1016/0021-9290(94)90276-3 — 38
* Prilutsky BI, Herzog W, Leonard T (1996). Transfer of mechanical energy between ankle and knee joints by gastrocnemius and plantaris muscles during cat locomotion. *J Biomech*. doi:10.1016/0021-9290(95)00054-2 — 56 (title/record only)
* Herzog W, Leonard TR (1991). Validation of optimization models that estimate the forces exerted by synergistic muscles. *J Biomech* 24 Suppl 1:31–39. doi:10.1016/0021-9290(91)90375-W — 160
* Herzog W, Leonard TR, Guimaraes AC (1993). Forces in gastrocnemius, soleus, and plantaris tendons of the freely moving cat. *J Biomech* 26:945–953. doi:10.1016/0021-9290(93)90056-K — 105
* Karabulut D, Dogru SC, Lin YC, Pandy MG, Herzog W, Arslan YZ (2020). Direct validation of model-predicted muscle forces in the cat hindlimb during locomotion. *J Biomech Eng* 142:051014. doi:10.1115/1.4045660 — 12
* Dul J, Johnson GE, Shiavi R, Townsend MA (1984). Muscular synergism II. A minimum-fatigue criterion for load sharing between synergistic muscles. *J Biomech* 17:675–684. doi:10.1016/0021-9290(84)90121-0 — 301

**Instrumented knee: Grand Challenge and follow-ups**

* Fregly BJ, Besier TF, Lloyd DG, Delp SL, Banks SA, Pandy MG, D'Lima DD (2012). Grand challenge competition to predict in vivo knee loads. *J Orthop Res* 30(4):503–513. doi:10.1002/jor.22023 — 622 (full text, PMC4067494)
* Kinney AL, Besier TF, D'Lima DD, Fregly BJ (2013). Update on grand challenge competition to predict in vivo knee loads. *J Biomech Eng* 135(2):021012. doi:10.1115/1.4023255 — 113 (full text, PMC3597120)
* D'Lima DD, Townsend CP, Arms SW, Morris BA, Colwell CW (2005). An implantable telemetry device to measure intra-articular tibial forces. *J Biomech* 38:299–304. doi:10.1016/j.jbiomech.2004.02.011 — 200
* Kirking B, Krevolin J, Townsend C, Colwell CW, D'Lima DD (2006). A multiaxial force-sensing implantable tibial prosthesis. *J Biomech* 39:1744–1751. doi:10.1016/j.jbiomech.2005.05.023 — 121
* Kim HJ, Fernandez JW, Akbarshahi M, Walter JP, Fregly BJ, Pandy MG (2009). Evaluation of predicted knee-joint muscle forces during gait using an instrumented knee implant. *J Orthop Res* 27:1326. doi:10.1002/jor.20876 — 194
* Lerner ZF, DeMers MS, Delp SL, Browning RC (2015). How tibiofemoral alignment and contact locations affect predictions of medial and lateral tibiofemoral contact forces. *J Biomech* 48:644. doi:10.1016/j.jbiomech.2014.12.049 — 228
* Walter JP, Kinney AL, Banks SA, D'Lima DD, Besier TF, Lloyd DG, Fregly BJ (2014). Muscle synergies may improve optimization prediction of knee contact forces during walking. *J Biomech Eng* 136:021031. doi:10.1115/1.4026428 — 87
* Serrancolí G, Kinney AL, Fregly BJ, Font-Llagunes JM (2016). Neuromusculoskeletal model calibration significantly affects predicted knee contact forces for walking. *J Biomech Eng* 138:081001. doi:10.1115/1.4033673 — 51
* Thelen DG, Won Choi K, Schmitz AM (2014). Co-simulation of neuromuscular dynamics and knee mechanics during human walking. *J Biomech Eng* 136:021033. doi:10.1115/1.4026358 — 102
* Imani Nejad Z, et al., Taylor WR, Smith CR (2020). The capacity of generic musculoskeletal simulations to predict knee joint loading using the CAMS-Knee datasets. *Ann Biomed Eng* 48:1430. doi:10.1007/s10439-020-02465-5 — 51
* Bennett KJ, Pizzolato C, Martelli S, et al. (2022). EMG-informed neuromusculoskeletal models accurately predict knee loading measured using instrumented implants. *IEEE Trans Biomed Eng* 69:2268. doi:10.1109/TBME.2022.3141067 — 53
* Trepczynski A, Kutzner I, Schwachmeyer V, Heller MO, Pfitzner T, Duda GN (2018). Impact of antagonistic muscle co-contraction on in vivo knee contact forces. *J NeuroEng Rehabil* 15:101. doi:10.1186/s12984-018-0434-3 — 68

**Static vs dynamic, hip, femur**

* Edwards WB, Miller RH, Derrick TR (2016). Femoral strain during walking predicted with muscle forces from static and dynamic optimization. *J Biomech* 49(7):1206–1213. doi:10.1016/j.jbiomech.2016.03.007 — 31 (full text, accepted manuscript doi:10.11575/PRISM/34235)
* Anderson FC, Pandy MG (2001). Static and dynamic optimization solutions for gait are practically equivalent. *J Biomech* 34:153–161. doi:10.1016/S0021-9290(00)00155-X — 815
* Wesseling M, et al., Jonkers I (2015). Muscle optimization techniques impact the magnitude of calculated hip joint contact forces. *J Orthop Res* 33:430–438. doi:10.1002/jor.22769 — 61
* Bergmann G, et al. (2001). Hip contact forces and gait patterns from routine activities. *J Biomech* 34:859. doi:10.1016/S0021-9290(01)00040-9 — 2297
* Erdemir A, McLean S, Herzog W, van den Bogert AJ (2007). Model-based estimation of muscle forces exerted during movements. *Clin Biomech* 22:131–154. doi:10.1016/j.clinbiomech.2006.09.005 — 925
* Aamodt A, Lund-Larsen J, Eine J, Andersen E, Benum P, Husby OS (1997). In vivo measurements show tensile axial strain in the proximal lateral aspect of the human femur. *J Orthop Res* 15:927–931. doi:10.1002/jor.1100150620 — 129
* Taylor SJG, Walker PS (2001). Forces and moments telemetered from two distal femoral replacements during various activities. *J Biomech* 34:839. doi:10.1016/S0021-9290(01)00042-2 — 220
* Duda GN, Heller M, Albinger J, Schulz O, Schneider E, Claes L (1998). Influence of muscle forces on femoral strain distribution. *J Biomech* 31:841–846. doi:10.1016/S0021-9290(98)00080-3 — 362
* Komi PV (1990). Relevance of in vivo force measurements to human biomechanics. *J Biomech* 23 Suppl 1:23. doi:10.1016/0021-9290(90)90038-5 — 395
* Finni T, Komi PV, Lukkariniemi J (1998). Achilles tendon loading during walking: application of a novel optic fiber technique. *Eur J Appl Physiol* 77:289. doi:10.1007/s004210050335 — 279
* Gregor RJ, Komi PV, Browning RC, Järvinen M (1991). A comparison of the triceps surae and residual muscle moments at the ankle during cycling. *J Biomech* 24:287. doi:10.1016/0021-9290(91)90347-P — 110